# Phase 1: BankSim Dataset Inspection

This notebook inspects the BankSim CSV at `D:/Fraud Detection/bs140513_032310.csv`; the source file stays outside the Git repository and is never modified or copied. The inspection reads the file in chunks, reports schema, row and quality counts, fraud balance, and streaming numeric summaries, then creates a reproducible 50,000-row sample stratified by time step and fraud label.

No behavioral or rolling features are created, and no modeling is performed. Run cells from top to bottom. The first inspection cell prints only the header and a five-row preview; later cells make bounded-memory passes over the source.

In [ ]:
from collections import Counter
from pathlib import Path
import json
import math
import sqlite3
import tempfile

import numpy as np
import pandas as pd

SOURCE_CSV = Path(r"D:\Fraud Detection\bs140513_032310.csv")
PROJECT_ROOT = Path.cwd()
OUTPUT_CSV = PROJECT_ROOT / "data" / "processed" / "banksim_phase1_sample_50000.csv"
CHUNK_SIZE = 100_000
SAMPLE_SIZE = 50_000
RANDOM_SEED = 42

if not SOURCE_CSV.is_file():
    raise FileNotFoundError(f"BankSim source CSV not found: {SOURCE_CSV}")

header = pd.read_csv(SOURCE_CSV, nrows=0).columns.tolist()
preview = pd.read_csv(SOURCE_CSV, nrows=5)
expected_columns = {
    "step", "customer", "age", "gender", "zipcodeOri",
    "merchant", "zipMerchant", "category", "amount", "fraud",
}
if set(header) != expected_columns:
    raise ValueError(
        f"Unexpected BankSim columns. Expected {sorted(expected_columns)}, got {header}"
    )

print(f"Source: {SOURCE_CSV}")
print(f"File size: {SOURCE_CSV.stat().st_size:,} bytes")
print("Header:", header)
display(preview)


## Load the complete dataset

This reads all rows for inspection; it does not create a reduced dataset. The original file remains untouched.

In [ ]:
df = pd.read_csv(DATASET_PATH, low_memory=False)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Shape: {df.shape}")
display(pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str).values}))

## Structure and data quality

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100),
}).sort_values("missing_count", ascending=False)
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
display(missing_summary)

reference_columns = {
    "step", "customer", "age", "gender", "zipcodeOri",
    "merchant", "zipMerchant", "category", "amount", "fraud",
}
actual_columns = set(df.columns)
print("Reference columns absent from this file:", sorted(reference_columns - actual_columns))
print("Columns present beyond the known BankSim schema:", sorted(actual_columns - reference_columns))

if "fraud" in df.columns:
    target_numeric = pd.to_numeric(df["fraud"], errors="coerce")
    fraud_mask = target_numeric.eq(1)
    genuine_mask = target_numeric.eq(0)
    invalid_target_mask = ~(fraud_mask | genuine_mask)
    fraud_count = int(fraud_mask.sum())
    genuine_count = int(genuine_mask.sum())
    print("Target value counts (including missing):")
    display(df["fraud"].value_counts(dropna=False).rename_axis("fraud").to_frame("count"))
    print(f"Fraud count (fraud == 1): {fraud_count:,}")
    print(f"Non-fraud count (fraud == 0): {genuine_count:,}")
    print(f"Fraud as percent of all rows: {fraud_count / len(df) * 100:.6f}%")
    print(f"Rows with missing or non-binary target values: {int(invalid_target_mask.sum()):,}")
else:
    print("No 'fraud' target column is present; target counts cannot be computed.")

## Categorical values, distributions, and cardinality

Low-cardinality columns show all observed values. High-cardinality identifiers show their full distinct count and the 20 most common values to keep the notebook readable.

In [ ]:
known_categorical = [
    "age", "gender", "zipcodeOri", "merchant",
    "zipMerchant", "category", "customer",
]
categorical_columns = list(dict.fromkeys(
    [column for column in known_categorical if column in df.columns]
    + df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
))
cardinality = pd.DataFrame({
    "unique_values": {column: df[column].nunique(dropna=False) for column in categorical_columns}
}).sort_values("unique_values", ascending=False)
display(cardinality)
for column in categorical_columns:
    counts = df[column].value_counts(dropna=False)
    print(f"{column}: {len(counts):,} observed values")
    display(counts.to_frame("count") if len(counts) <= 30 else counts.head(20).to_frame("count (top 20)"))

## Transaction amounts and fraud comparisons

In [ ]:
if "amount" in df.columns:
    amount_numeric = pd.to_numeric(df["amount"], errors="coerce")
    print("Amount statistics (all rows):")
    display(amount_numeric.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame("amount"))
    print(f"Missing or non-numeric amounts: {int(amount_numeric.isna().sum()):,}")
    print(f"Negative amounts: {int(amount_numeric.lt(0).sum()):,}")
    if "fraud" in df.columns:
        amount_by_class = pd.DataFrame({
            "amount": amount_numeric,
            "class": pd.Series("other/invalid", index=df.index),
        })
        amount_by_class.loc[genuine_mask, "class"] = "genuine (fraud=0)"
        amount_by_class.loc[fraud_mask, "class"] = "fraud (fraud=1)"
        display(amount_by_class.groupby("class")["amount"].describe(
            percentiles=[0.25, 0.5, 0.75, 0.95]
        ))
else:
    print("No 'amount' column is present.")

In [ ]:
if "amount" in df.columns:
    import matplotlib.pyplot as plt

    values = amount_numeric.dropna()
    plt.figure(figsize=(9, 4))
    plt.hist(values, bins=50, color="steelblue", edgecolor="white")
    plt.title("Transaction amount distribution (50 bins)")
    plt.xlabel("Amount")
    plt.ylabel("Transaction count")
    plt.tight_layout()
    plt.show()

## Time coverage

The `step` field is reported in its observed units. No calendar date is fabricated when the file does not provide a calendar epoch. Actual datetime-like columns, if present, are parsed for a range.

In [ ]:
if "step" in df.columns:
    step_numeric = pd.to_numeric(df["step"], errors="coerce")
    print(f"step minimum: {step_numeric.min()}")
    print(f"step maximum: {step_numeric.max()}")
    print(f"Distinct non-missing step values: {step_numeric.nunique():,}")
    step_counts = step_numeric.value_counts().sort_index()
    display(step_counts.rename_axis("step").to_frame("transaction_count"))
else:
    print("No 'step' column is present.")

datetime_columns = [
    column for column in df.columns
    if any(token in column.lower() for token in ("date", "timestamp", "datetime"))
]
if datetime_columns:
    for column in datetime_columns:
        parsed = pd.to_datetime(df[column], errors="coerce", utc=True)
        print(f"{column}: parsed {int(parsed.notna().sum()):,} of {len(df):,}; range {parsed.min()} to {parsed.max()}")
else:
    print("No calendar date/timestamp column detected.")

## Fraud distribution by category, gender, and age

In [ ]:
if "fraud" in df.columns:
    fraud_indicator = fraud_mask.astype("int64")
    for column in ("category", "gender", "age"):
        if column in df.columns:
            group_frame = pd.DataFrame({column: df[column], "is_fraud": fraud_indicator})
            summary = group_frame.groupby(column, dropna=False).agg(
                transaction_count=("is_fraud", "size"),
                fraud_count=("is_fraud", "sum"),
            )
            summary["fraud_percent"] = summary["fraud_count"].div(summary["transaction_count"]).mul(100)
            print(f"Fraud by {column}:")
            display(summary.sort_values("transaction_count", ascending=False))

    for column in ("zipcodeOri", "zipMerchant"):
        if column in df.columns:
            print(f"{column} cardinality: {df[column].nunique(dropna=False):,}")
            display(df[column].value_counts(dropna=False).head(20).to_frame("count (top 20)"))

## Unexpected values and data-quality flags

In [ ]:
if "fraud" in df.columns:
    unexpected_target_values = df.loc[invalid_target_mask, "fraud"].value_counts(dropna=False)
    print("Unexpected/missing target values beyond numeric 0 and 1:")
    display(unexpected_target_values.to_frame("count"))

if "age" in df.columns:
    allowed_age_codes = {"0", "1", "2", "3", "4", "5", "6", "U"}
    observed_age_codes = set(df["age"].dropna().astype(str).str.strip().str.upper().unique())
    print("Age values outside the documented codes 0-6 and U:", sorted(observed_age_codes - allowed_age_codes))
if "gender" in df.columns:
    allowed_gender_codes = {"F", "M", "E"}
    observed_gender_codes = set(df["gender"].dropna().astype(str).str.strip().str.upper().unique())
    print("Gender values outside the documented codes F, M, and E:", sorted(observed_gender_codes - allowed_gender_codes))
if "amount" in df.columns:
    print(f"Non-finite numeric amounts: {int((~amount_numeric.dropna().map(lambda value: __import__('math').isfinite(value))).sum()):,}")

## Proposed 50,000-row sampling strategy (not executed)

Do not take the first 50,000 rows or use an unstratified random sample. After reviewing the observed class counts and time coverage, reserve the latest chronological period for testing. Within the earlier period, divide the observed time range into ordered bins and sample reproducibly within each time-bin/class stratum using a fixed seed (for example, `random_state=42`). Allocate the 50,000-row budget across those strata based on actual counts, while setting a defensible fraud minimum only if the observed data supports it. Keep validation earlier than test, retain chronological ordering after selection, and document each selected period and class count.

This Phase 1 notebook does not create a sample or write any dataset files.